# autoqmsp: automated qMSP analysis in Google Colab

Analyse QuantStudio / Applied Biosystems **.eds** files in your browser. You don't need to install anything.

**How to use**
1. Check that the runtime is **R**: *Runtime → Change runtime type → R*. This notebook should open in R already.
2. Run cell 1 to install the package. It takes about 1 minute, once per session.
3. Upload your `.eds` files: click the **folder icon** on the left, then drag and drop the files (they go to `/content`).
4. Check the settings in cell 3, then run all the remaining cells.
5. Download `qmsp_results.xlsx` from the file panel: right-click it and choose **Download**.

Uploaded files are deleted when the Colab session ends.

## 1. Install

In [ ]:
if (!requireNamespace("remotes", quietly = TRUE)) install.packages("remotes")
if (!requireNamespace("writexl", quietly = TRUE)) install.packages("writexl")
remotes::install_github("nazliakilli2/Automated-qMSP", upgrade = "never", quiet = TRUE)
library(autoqmsp)
options(repr.plot.width = 14, repr.plot.height = 9, width = 200)

## 2. Read the .eds files
Reads every `.eds` file uploaded to `/content`.

In [ ]:
files <- list.files("/content", pattern = "\\.eds$", full.names = TRUE, ignore.case = TRUE)
if (!length(files)) stop("No .eds files found - upload them with the folder icon on the left first.")
runs <- read_eds_files(files, names = tools::file_path_sans_ext(basename(files)))
runs

## 3. Settings
- `reference`: name of the reference gene (a regular expression, not case-sensitive). Use `NULL` if the run has no reference gene.
- `ntc` / `positive`: patterns that identify the no-template controls and the positive (methylated) controls by sample name.
- `ct_cutoff`: a gene with Ct at or below this value is called **methylated**.
- `ref_ct_max`: a sample whose reference-gene Ct is above this value (or undetermined) is **invalid**. Above `ref_ct_warn`, it is flagged as low DNA input.
- `min_cq_conf` / `min_plateau`: an amplified well with a low instrument confidence, or with a curve much lower than the positive control's, is sent to **Review** instead of being called.

In [ ]:
res <- analyze_qmsp(
  runs,
  reference   = "ACTB|B.?ACTIN|BETA.?ACTIN",
  ntc         = "NTC|dH2O|dH20|water|blank|^NK",
  positive    = "H460|A549|HT29|positive|^PC\\b",
  ct_cutoff   = 40,
  ref_ct_max  = 40,
  ref_ct_warn = 35,
  min_cq_conf = 0.5,
  min_plateau = 0.2
)
res

## 4. Summary: one row per sample, one column per gene

In [ ]:
results_wide(res, "call")
plot_methylation(res)

## 5. Controls and wells that need a manual look

In [ ]:
res$controls
res$wells[res$wells$result == "Review", c("run", "well", "sample", "target", "ct", "cq_conf", "flags")]

## 6. Full results (Ct, reference Ct, ΔCt, ratio, PMR, notes)

In [ ]:
r <- res$results
r[, c("run", "sample", "role", "target", "ct", "ref_ct", "ref_status", "delta_ct", "ratio", "pmr", "call", "notes")]

## 7. Amplification curves and plate map for each run

In [ ]:
for (run in unique(res$wells$run)) {
  print(plot_amplification(res, run = run))
  print(plot_plate(res, run = run))
}

## 8. Export to Excel
Then download `qmsp_results.xlsx` from the file panel on the left.

In [ ]:
export_results(res, "/content/qmsp_results.xlsx")